In [1]:
import io
import time
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from pathlib import Path

# ==========================================
# SETUP & CONFIGURATION
# ==========================================
TEST_MODE = False        # <--- TOGGLE THIS: True = 3 stations, False = All stations
TEST_LIMIT = 3          # Number of stations to run in test mode

RAW_HOURLY = Path("raw_noaa_hourly")
RAW_HOURLY.mkdir(exist_ok=True)
RAW_DAILY = Path("raw_noaa_daily")
RAW_DAILY.mkdir(exist_ok=True)
OUT = Path("weather_downloads")
OUT.mkdir(exist_ok=True)
session = requests.Session()

START_YEAR = 2002
END_YEAR = 2026
LATEST_DATE = pd.Timestamp("2026-10-19")

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def clean_column_name(col):
    return str(col).strip().lower().replace(" ", "_").replace("-", "_")

def find_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns: return candidate
    return None

def find_flag_column(df, base_col_name):
    # NOAA usually appends flags with _quality_control, _qc, _flag, or _attributes
    if not base_col_name: return None
    for col in df.columns:
        if col.startswith(base_col_name) and any(x in col for x in ['quality', 'flag', 'qc', 'attribute']):
            return col
    return None

def calc_specific_humidity(t_celsius, rh_percent, p_hpa):
    """
    Calculates Specific Humidity (kg/kg) from Temp, RH, and Pressure.
    Ignores -9999 missing values to prevent math errors.
    """
    q = np.full_like(t_celsius, -9999.0, dtype=float)
    valid = (t_celsius > -100) & (rh_percent >= 0) & (p_hpa > 0)
    
    t = t_celsius[valid]
    rh = rh_percent[valid]
    p = p_hpa[valid]
    
    # Calculate saturation vapor pressure (es) in hPa
    es = 6.112 * np.exp((17.67 * t) / (t + 243.5))
    # Actual vapor pressure (e) in hPa
    e = es * (rh / 100.0)
    # Specific humidity (kg/kg)
    q[valid] = (0.622 * e) / (p - (0.378 * e))
    
    return q

# ==========================================
# 1. FIND STATIONS VIA BOUNDING BOXES
# ==========================================
print("Fetching master station list from NOAA...")
station_url = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh-station-list.txt"
r = session.get(station_url)
r.raise_for_status()

colspecs = [(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)]
names = ["station_id", "latitude", "longitude", "elevation", "state", "name"]
all_stations = pd.read_fwf(io.StringIO(r.text), colspecs=colspecs, names=names)

all_stations = all_stations[all_stations['station_id'].str.startswith('USW', na=False)]

bounding_boxes = {
    "va": (38.5, 39.5, -78.0, -77.0),
    "tx": (32.3, 33.3, -97.5, -96.5)
}

selected_stations = []
for region, (min_lat, max_lat, min_lon, max_lon) in bounding_boxes.items():
    mask = (
        (all_stations["latitude"] >= min_lat) & (all_stations["latitude"] <= max_lat) &
        (all_stations["longitude"] >= min_lon) & (all_stations["longitude"] <= max_lon)
    )
    region_stations = all_stations[mask].copy()
    region_stations["region"] = region
    selected_stations.append(region_stations)

selected = pd.concat(selected_stations, ignore_index=True)

# APPLY TEST MODE TOGGLE
if TEST_MODE:
    print(f"\n*** TEST MODE ENABLED: Limiting to {TEST_LIMIT} stations ***")
    selected = selected.head(TEST_LIMIT)

print(f"\nFound {len(selected)} high-quality stations to process:")
for idx, row in selected.iterrows():
    print(f" - {row['station_id']}: {row['name']} ({row['region']})")

# ==========================================
# 2. DOWNLOAD HOURLY DATA (GHCNh)
# ==========================================
print("\n--- Starting HOURLY Data Pull ---")
HOURLY_BASE = "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/{year}/psv/GHCNh_{station}_{year}.psv"

hourly_data = []

for _, station in tqdm(selected.iterrows(), total=len(selected), desc="Hourly Stations"):
    station_id = str(station["station_id"]).strip()

    for year in range(START_YEAR, END_YEAR + 1):
        url = HOURLY_BASE.format(year=year, station=station_id)
        local_file = RAW_HOURLY / f"GHCNh_{station_id}_{year}.psv"

        try:
            if local_file.exists() and local_file.stat().st_size > 0:
                content = local_file.read_bytes()
            else:
                r = session.get(url, timeout=90)
                if r.status_code == 404: continue
                r.raise_for_status()
                content = r.content
                local_file.write_bytes(content)

            df = pd.read_csv(io.BytesIO(content), sep="|", dtype=str, low_memory=False)
            df.columns = [clean_column_name(c) for c in df.columns]

            year_col, month_col = find_column(df, ["year"]), find_column(df, ["month"])
            day_col, hour_col = find_column(df, ["day"]), find_column(df, ["hour"])
            minute_col = find_column(df, ["minute"])

            if not all([year_col, month_col, day_col, hour_col]): continue

            date_parts = (
                df[year_col].str.zfill(4) + df[month_col].str.zfill(2) + 
                df[day_col].str.zfill(2) + df[hour_col].str.zfill(2) + 
                (df[minute_col].str.zfill(2) if minute_col else "00")
            )
            df["datetime"] = pd.to_datetime(date_parts, format="%Y%m%d%H%M", errors="coerce", utc=True)

            # Find main columns
            temp_col = find_column(df, ["temperature", "dry_bulb_temperature"])
            rh_col = find_column(df, ["relative_humidity", "rh"])
            dew_col = find_column(df, ["dew_point_temperature", "dew_point"])
            pressure_col = find_column(df, ["station_level_pressure"]) # "sea_level_pressure", "altimeter"])

            if temp_col is None or rh_col is None or pressure_col is None: continue

            # Find flag columns dynamically
            temp_flag = find_flag_column(df, temp_col)
            rh_flag = find_flag_column(df, rh_col)
            pressure_flag = find_flag_column(df, pressure_col)

            part = pd.DataFrame({
                "datetime": df["datetime"],
                "station_id": station_id,
                "station_name": station["name"],
                "region": station["region"],
                "latitude": station["latitude"],
                "longitude": station["longitude"],
                "temperature": pd.to_numeric(df[temp_col], errors="coerce"),
                "temp_flag": df[temp_flag] if temp_flag else None,
                "dew_point": pd.to_numeric(df[dew_col], errors="coerce") if dew_col else -9999,
                "relative_humidity": pd.to_numeric(df[rh_col], errors="coerce"),
                "rh_flag": df[rh_flag] if rh_flag else None,
                "station_pressure": pd.to_numeric(df[pressure_col], errors="coerce"),
                "pressure_flag": df[pressure_flag] if pressure_flag else None,
            })
            
            # Calculate Specific Humidity (kg/kg)
            # Note: Station pressure in GHCNh is often hPa, but occasionally converted. Assuming hPa for standard formula.
            part["specific_humidity_calculated"] = calc_specific_humidity(
                part["temperature"].values, 
                part["relative_humidity"].values, 
                part["station_pressure"].values
            )

            # Filter dates, DO NOT filter out -9999 values per user request
            part = part[part["datetime"].notna() & (part["datetime"].dt.year >= START_YEAR) & (part["datetime"] <= LATEST_DATE + pd.Timedelta(days=1))]
            
            if not part.empty:
                hourly_data.append(part)

        except Exception as exc:
            pass # Silently skip errors in test mode
        time.sleep(0.15)

if hourly_data:
    weather_h = pd.concat(hourly_data, ignore_index=True)
    weather_h = weather_h.drop_duplicates(["station_id", "datetime"]).sort_values(["region", "station_id", "datetime"])
    weather_h.to_csv(OUT / "noaa_hourly_observations.csv", index=False)
    print(f"Saved {len(weather_h):,} HOURLY observations (with flags and specific humidity).")

# ==========================================
# 3. DOWNLOAD DAILY DATA (GHCNd)
# ==========================================
print("\n--- Starting DAILY Data Pull ---")
DAILY_BASE = "https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/{station}.csv"

daily_data = []

for _, station in tqdm(selected.iterrows(), total=len(selected), desc="Daily Stations"):
    station_id = str(station["station_id"]).strip()
    url = DAILY_BASE.format(station=station_id)
    local_file = RAW_DAILY / f"GHCNd_{station_id}.csv"

    try:
        if local_file.exists() and local_file.stat().st_size > 0:
            df = pd.read_csv(local_file, dtype=str, low_memory=False)
        else:
            r = session.get(url, timeout=90)
            if r.status_code == 404: continue
            r.raise_for_status()
            local_file.write_bytes(r.content)
            df = pd.read_csv(io.BytesIO(r.content), dtype=str, low_memory=False)

        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
        df = df[(df["DATE"].dt.year >= START_YEAR) & (df["DATE"] <= LATEST_DATE)].copy()

        if df.empty: continue

        part_d = pd.DataFrame({
            "date": df["DATE"],
            "station_id": station_id,
            "station_name": station["name"],
            "region": station["region"],
            "latitude": station["latitude"],
            "longitude": station["longitude"],
        })

        # Pull Daily columns and their Quality Flag attributes
        if "TMAX" in df.columns: 
            part_d["tmax"] = pd.to_numeric(df["TMAX"], errors="coerce")
            part_d["tmax_flag"] = df["TMAX_ATTRIBUTES"] if "TMAX_ATTRIBUTES" in df.columns else None
        if "TMIN" in df.columns: 
            part_d["tmin"] = pd.to_numeric(df["TMIN"], errors="coerce")
            part_d["tmin_flag"] = df["TMIN_ATTRIBUTES"] if "TMIN_ATTRIBUTES" in df.columns else None
        if "PRCP" in df.columns: 
            part_d["precip"] = pd.to_numeric(df["PRCP"], errors="coerce")
            part_d["precip_flag"] = df["PRCP_ATTRIBUTES"] if "PRCP_ATTRIBUTES" in df.columns else None

        daily_data.append(part_d)
        
    except Exception as exc:
        pass
    time.sleep(0.5)

if daily_data:
    weather_d = pd.concat(daily_data, ignore_index=True)
    weather_d = weather_d.sort_values(["region", "station_id", "date"])
    weather_d.to_csv(OUT / "noaa_daily_observations.csv", index=False)
    print(f"Saved {len(weather_d):,} DAILY observations (with flags).")

print("\nAll downloads complete.")

Fetching master station list from NOAA...

Found 38 high-quality stations to process:
 - USW00000367: WARRENTON FAUQUIER AP (va)
 - USW00003710: MANASSAS RGNL AP (va)
 - USW00003714: LEESBURG EXECUTIVE AP (va)
 - USW00013710: WASHINGTON DC BOLLING FLD AFB (va)
 - USW00013730: FREDERICK MUNI AP (va)
 - USW00013734: MARTINSBURG E W VIRGINIA RGNL (va)
 - USW00013743: WASHINGTON REAGAN NATL AP (va)
 - USW00013749: FREDERICK AF (va)
 - USW00013751: ANACOSTIA NAS (va)
 - USW00013773: QUANTICO MCAS (va)
 - USW00093725: WASHINGTON DC WB CITY (va)
 - USW00093728: DAVISON AAF (va)
 - USW00093734: STERLING (va)
 - USW00093738: WASHINGTON DULLES INTL AP (va)
 - USW00093764: GAITHERSBURG MONTGOMERY CO AP (va)
 - USW00093798: CULPEPER RGNL AP (va)
 - USW00003927: DAL-FTW WSCMO AP (tx)
 - USW00003970: DALLAS ADDISON AP (tx)
 - USW00003971: DALLAS REDBIRD AP (tx)
 - USW00003985: FORT WORTH SPINKS AP (tx)
 - USW00003991: DENTON MUNICPAL AP (tx)
 - USW00013907: DALLAS HENSLEY FLD AFB (tx)
 - USW00013911

Hourly Stations: 100%|██████████| 38/38 [20:51<00:00, 32.95s/it]



--- Starting DAILY Data Pull ---


Daily Stations: 100%|██████████| 38/38 [00:21<00:00,  1.75it/s]


Saved 143,447 DAILY observations (with flags).

All downloads complete.
